In [1]:
# 1. Install required packages
!pip -q install fastapi uvicorn pandas scikit-learn joblib requests nest-asyncio pyngrok

In [2]:
# 2. Upload the kidney disease CSV dataset
from google.colab import files
uploaded = files.upload()

import os
uploaded_csv = [name for name in uploaded if name.lower().endswith(".csv")]
if not uploaded_csv:
    raise FileNotFoundError("Please upload the kidney disease CSV file.")
CSV_PATH = uploaded_csv[0]
print("Using dataset:", CSV_PATH)

Saving 1. kidney_disease.csv to 1. kidney_disease.csv
Using dataset: 1. kidney_disease.csv


In [3]:
# 3. Inspect and prepare the dataset
import pandas as pd
import numpy as np

df = pd.read_csv(CSV_PATH)
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df = df.drop(columns=["id"], errors="ignore")
target_col = "classification"
if target_col not in df.columns:
    raise ValueError(f"Expected target column '{target_col}'. Found: {df.columns.tolist()}")

df[target_col] = (
    df[target_col].astype(str).str.strip().str.lower()
      .str.replace(r"\\s+", "", regex=True)
      .replace({"notckd": "notckd", "ckd\\t": "ckd"})
)
print("Target distribution:")
display(df[target_col].value_counts(dropna=False))

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


Dataset shape: (400, 26)
Columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']
Target distribution:


,count
classification,
ckd,250
notckd,150


In [4]:
# 4. Train and save a reusable preprocessing + classification pipeline
# NOTE: If you already have the exact Session 1 kidney_disease_model.pkl,
# upload it and skip this cell to comply with "do not retrain" instructions.

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df.drop(columns=[target_col])
y = df[target_col]

numeric_cols = [c for c in ["age","bp","sg","al","su","bgr","bu","sc","sod","pot","hemo"] if c in X.columns]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# Evaluate on a held-out split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
evaluation_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250, random_state=42, class_weight="balanced"
    ))
])
evaluation_model.fit(X_train, y_train)
y_pred = evaluation_model.predict(X_test)
print("Hold-out accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, zero_division=0))

# Train final serving pipeline once on all labelled rows and save it
serving_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250, random_state=42, class_weight="balanced"
    ))
])
serving_model.fit(X, y)
MODEL_PATH = "/content/kidney_disease_model.pkl"
joblib.dump(serving_model, MODEL_PATH)
print("Saved model:", MODEL_PATH)

Hold-out accuracy: 0.9875
              precision    recall  f1-score   support

         ckd       0.98      1.00      0.99        50
      notckd       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80

Saved model: /content/kidney_disease_model.pkl


## 5. FastAPI application

The API below exposes:
- `GET /` — application status
- `GET /health` — health check
- `POST /predict` — validated prediction
- Swagger UI at `/docs`

If you have the original Session 1 model, upload it to `/content/kidney_disease_model.pkl` before starting the server.

In [5]:
# 5. Create app.py
%%writefile /content/app.py
from pathlib import Path
from typing import Optional, Any
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

MODEL_PATH = Path("/content/kidney_disease_model.pkl")
FEATURES = [
    "age", "bp", "sg", "al", "su", "rbc", "pc", "pcc", "ba",
    "bgr", "bu", "sc", "sod", "pot", "hemo", "pcv", "wc", "rc",
    "htn", "dm", "cad", "appet", "pe", "ane"
]
model = joblib.load(MODEL_PATH)

app = FastAPI(
    title="Kidney Disease Prediction API",
    description="Educational MLOps model-serving demonstration.",
    version="1.0.0"
)

class KidneyDiseaseInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    age: Optional[float] = None
    bp: Optional[float] = None
    sg: Optional[float] = None
    al: Optional[float] = None
    su: Optional[float] = None
    rbc: Optional[str] = None
    pc: Optional[str] = None
    pcc: Optional[str] = None
    ba: Optional[str] = None
    bgr: Optional[float] = None
    bu: Optional[float] = None
    sc: Optional[float] = None
    sod: Optional[float] = None
    pot: Optional[float] = None
    hemo: Optional[float] = None
    pcv: Optional[str] = None
    wc: Optional[str] = None
    rc: Optional[str] = None
    htn: Optional[str] = None
    dm: Optional[str] = None
    cad: Optional[str] = None
    appet: Optional[str] = None
    pe: Optional[str] = None
    ane: Optional[str] = None

@app.get("/")
def root():
    return {"message": "Kidney Disease ML API is running", "version": "1.0.0"}

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/predict")
def predict(data: KidneyDiseaseInput) -> dict[str, Any]:
    try:
        record = data.model_dump()
        frame = pd.DataFrame([[record.get(col) for col in FEATURES]], columns=FEATURES)
        prediction = model.predict(frame)[0]
        response = {"prediction": str(prediction)}
        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(frame)[0]
            response["probabilities"] = {
                str(label): round(float(prob), 4)
                for label, prob in zip(model.classes_, probabilities)
            }
        response["disclaimer"] = "Educational demonstration only; not a clinical diagnosis."
        return response
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {type(exc).__name__}: {exc}") from exc


Writing /content/app.py


In [6]:
# 6. Start FastAPI in the Colab runtime
import nest_asyncio, subprocess, time, requests, os, signal
nest_asyncio.apply()

# Stop an old server process if this cell is run again
try:
    server_process.terminate()
except Exception:
    pass

server_process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)
print("Server process started:", server_process.poll() is None)
print("Local health response:", requests.get("http://127.0.0.1:8000/health", timeout=10).json())

Server process started: True
Local health response: {'status': 'healthy', 'model_loaded': True}


### 7. Expose the API for Swagger testing

Colab runtimes are remote. To open Swagger from your browser, create a temporary public tunnel. Run the next cell and follow its URL. If the tunnel asks for an authentication token, add your ngrok authtoken in the cell as described in the ngrok dashboard.

In [7]:
# 7. Create a temporary ngrok tunnel (optional, for browser-based Swagger access)
# Set NGROK_AUTH_TOKEN in Colab Secrets or paste your token below if your account requires it.
from pyngrok import ngrok

NGROK_AUTH_TOKEN = ""  # Optional: paste your ngrok auth token here
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

try:
    public_tunnel = ngrok.connect(8000, "http")
    PUBLIC_URL = public_tunnel.public_url
    print("Public API URL:", PUBLIC_URL)
    print("Swagger UI:", PUBLIC_URL + "/docs")
except Exception as e:
    PUBLIC_URL = None
    print("Tunnel could not be created:", e)
    print("You can still test locally in the next cell.")

ERROR:pyngrok.process.ngrok:t=2026-10-24T03:31:36+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-24T03:31:36+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


Tunnel could not be created: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
You can still test locally in the next cell.


In [8]:
# 8. Test GET / and GET /health
BASE_URL = PUBLIC_URL if PUBLIC_URL else "http://127.0.0.1:8000"
print("GET /:", requests.get(BASE_URL + "/", timeout=20).status_code,
      requests.get(BASE_URL + "/", timeout=20).json())
print("GET /health:", requests.get(BASE_URL + "/health", timeout=20).status_code,
      requests.get(BASE_URL + "/health", timeout=20).json())

GET /: 200 {'message': 'Kidney Disease ML API is running', 'version': '1.0.0'}
GET /health: 200 {'status': 'healthy', 'model_loaded': True}


In [9]:
# 9. Test POST /predict with a valid JSON record
sample = {
    "age": 48, "bp": 80, "sg": 1.02, "al": 1, "su": 0,
    "rbc": "normal", "pc": "normal", "pcc": "notpresent",
    "ba": "notpresent", "bgr": 121, "bu": 36, "sc": 1.2,
    "sod": 138, "pot": 4.5, "hemo": 15.4, "pcv": "44",
    "wc": "7800", "rc": "5.2", "htn": "no", "dm": "no",
    "cad": "no", "appet": "good", "pe": "no", "ane": "no"
}
response = requests.post(BASE_URL + "/predict", json=sample, timeout=30)
print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'prediction': 'notckd', 'probabilities': {'ckd': 0.428, 'notckd': 0.572}, 'disclaimer': 'Educational demonstration only; not a clinical diagnosis.'}


In [10]:
# 10. Test invalid input — should return HTTP 422
invalid_sample = dict(sample)
invalid_sample["age"] = "hello"
invalid_response = requests.post(BASE_URL + "/predict", json=invalid_sample, timeout=20)
print("Invalid request status (expected 422):", invalid_response.status_code)
print(invalid_response.text)

Invalid request status (expected 422): 422
{"detail":[{"type":"float_parsing","loc":["body","age"],"msg":"Input should be a valid number, unable to parse string as a number","input":"hello"}]}


In [11]:
# 11. Python client test (equivalent to client.py)
client_health = requests.get(BASE_URL + "/health", timeout=20)
client_prediction = requests.post(BASE_URL + "/predict", json=sample, timeout=30)
print("Client health:", client_health.status_code, client_health.json())
print("Client prediction:", client_prediction.status_code, client_prediction.json())

Client health: 200 {'status': 'healthy', 'model_loaded': True}
Client prediction: 200 {'prediction': 'notckd', 'probabilities': {'ckd': 0.428, 'notckd': 0.572}, 'disclaimer': 'Educational demonstration only; not a clinical diagnosis.'}


In [13]:
# 12. Save the model artifact and API client files for download
from google.colab import files
import shutil

# Save a separate client.py file in the Colab workspace
client_source = '''import requests

BASE_URL = "http://127.0.0.1:8000"
sample = ''' + repr({
    "age": 48, "bp": 80, "sg": 1.02, "al": 1, "su": 0,
    "rbc": "normal", "pc": "normal", "pcc": "notpresent",
    "ba": "notpresent", "bgr": 121, "bu": 36, "sc": 1.2,
    "sod": 138, "pot": 4.5, "hemo": 15.4, "pcv": "44",
    "wc": "7800", "rc": "5.2", "htn": "no", "dm": "no",
    "cad": "no", "appet": "good", "pe": "no", "ane": "no"
}) + '''

print("Health:", requests.get(BASE_URL + "/health").json())
response = requests.post(BASE_URL + "/predict", json=sample)
print("Status:", response.status_code)
print("Prediction:", response.json())
'''
with open("/content/client.py", "w") as f:
    f.write(client_source)

print("Created: /content/app.py, /content/client.py, /content/kidney_disease_model.pkl")
print("Use the Files pane in Colab to download them, or uncomment a download line below.")
# files.download("/content/kidney_disease_model.pkl")

Created: /content/app.py, /content/client.py, /content/kidney_disease_model.pkl
Use the Files pane in Colab to download them, or uncomment a download line below.
